Business Intelligence on physical retail outlets in Singapore

Target audience: Policymakers + Business owners

Workflow:
Collate sample from commercialguru dataset\
Exploratory data analysis on dataset\

Synthetic variables:\
footfall score\
floor score\
potential\
revenue_psf\
revenue_month

In [ ]:
# variable     | unit
# size         | sqft
# rent_psf     | $/sqft
# rent_month   | $
# mrt_distance | metres

In [ ]:
import pandas as pd
import numpy as np

excel_file = "commercialguru1.xlsx" 

# Read all sheets; first row in each sheet is used as column headers
region_data = pd.read_excel(excel_file, sheet_name=None, header=0)

# Show first 5 rows for each region (sheet)
for region, df in region_data.items():
    print(f"\n=== {region} ===")
    display(df.head(5))

## Analyse mean, median, quartile, min, max of rent by region


In [ ]:
# analyse mean, median, quartile, min, max of rent by region (column: rent_month)
rent_stats_rows = []

for region, df in region_data.items():
    if "rent_month" not in df.columns:
        rent_stats_rows.append(
            {
                "region": region,
                "count": 0,
                "min_rent": None,
                "q1_rent": None,
                "median_rent": None,
                "q3_rent": None,
                "max_rent": None,
                "mean_rent": None
            }
        )
        continue

    rent_numeric = pd.to_numeric(
        df["rent_month"]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("$", "", regex=False)
        .str.strip(),
        errors="coerce",
    ).dropna()

    if rent_numeric.empty:
        rent_stats_rows.append(
            {
                "region": region,
                "count": 0,
                "min_rent": None,
                "q1_rent": None,
                "median_rent": None,
                "q3_rent": None,
                "max_rent": None,
                "mean_rent": None
            }
        )
        continue

    rent_stats_rows.append(
        {
            "region": region,
            "count": int(rent_numeric.count()),
            "min_rent": round(rent_numeric.min(), 2),
            "q1_rent": round(rent_numeric.quantile(0.25), 2),
            "median_rent": round(rent_numeric.median(), 2),
            "q3_rent": round(rent_numeric.quantile(0.75), 2),
            "max_rent": round(rent_numeric.max(), 2),
            "mean_rent": round(rent_numeric.mean(), 2)
        }
    )

rent_stats_df = pd.DataFrame(rent_stats_rows).sort_values("region").reset_index(drop=True)
display(rent_stats_df)

## compare CBD vs heartland by tier using rent_month


In [ ]:
# compare CBD vs heartland by tier using rent_month variable

tier_rows = []

for region, df in region_data.items():
    if "tier" not in df.columns or "rent_month" not in df.columns:
        continue

    temp = df[["tier", "rent_month"]].copy()
    temp["region"] = region

    temp["rent_month_num"] = pd.to_numeric(
        temp["rent_month"]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("$", "", regex=False)
        .str.strip(),
        errors="coerce",
    )

    temp["tier_group"] = temp["tier"].astype(str).str.strip().str.lower()
    temp.loc[temp["tier_group"].str.contains("cbd", na=False), "tier_group"] = "CBD"
    temp.loc[temp["tier_group"].str.contains("heartland", na=False), "tier_group"] = "Heartland"

    tier_rows.append(temp)

if tier_rows:
    tier_df = pd.concat(tier_rows, ignore_index=True)
    tier_df = tier_df[tier_df["tier_group"].isin(["CBD", "Heartland"])]
    tier_df = tier_df.dropna(subset=["rent_month_num"])

    tier_rent_stats = (
        tier_df.groupby("tier_group")["rent_month_num"]
        .agg(count="count", mean="mean", median="median", min="min", max="max")
        .round(2)
        .reset_index()
        .sort_values("tier_group")
    )

    display(tier_rent_stats)
else:
    print("No usable 'tier' and 'rent_month' columns found in region_data.")

## analyse rent_psf by mean, median, quartile, min, max by region


In [ ]:
# analyse rent_psf by mean, median, quartile, min, max by region

rent_psf_stats_rows = []

for region, df in region_data.items():
    if "rent_psf" not in df.columns:
        rent_psf_stats_rows.append(
            {
                "region": region,
                "count": 0,
                "min_rent_psf": None,
                "q1_rent_psf": None,
                "median_rent_psf": None,
                "q3_rent_psf": None,
                "max_rent_psf": None,
                "mean_rent_psf": None,
            }
        )
        continue

    rent_psf_numeric = pd.to_numeric(
        df["rent_psf"]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("$", "", regex=False)
        .str.strip(),
        errors="coerce",
    ).dropna()

    if rent_psf_numeric.empty:
        rent_psf_stats_rows.append(
            {
                "region": region,
                "count": 0,
                "min_rent_psf": None,
                "q1_rent_psf": None,
                "median_rent_psf": None,
                "q3_rent_psf": None,
                "max_rent_psf": None,
                "mean_rent_psf": None,
            }
        )
        continue

    rent_psf_stats_rows.append(
        {
            "region": region,
            "count": int(rent_psf_numeric.count()),
            "min_rent_psf": round(rent_psf_numeric.min(), 2),
            "q1_rent_psf": round(rent_psf_numeric.quantile(0.25), 2),
            "median_rent_psf": round(rent_psf_numeric.median(), 2),
            "q3_rent_psf": round(rent_psf_numeric.quantile(0.75), 2),
            "max_rent_psf": round(rent_psf_numeric.max(), 2),
            "mean_rent_psf": round(rent_psf_numeric.mean(), 2),
        }
    )

rent_psf_stats_df = pd.DataFrame(rent_psf_stats_rows).sort_values("region").reset_index(drop=True)
display(rent_psf_stats_df)

## analyse correlation between size and rent_psf for all units


In [ ]:
# analyse correlation between size and rent_psf for all units
import scipy.stats as stats

all_units = []

for region, df in region_data.items():
    if "size" not in df.columns or "rent_psf" not in df.columns:
        continue

    temp = df[["size", "rent_psf"]].copy()
    temp["region"] = region

    temp["size_num"] = pd.to_numeric(
        temp["size"].astype(str).str.replace(",", "", regex=False).str.strip(),
        errors="coerce",
    )
    temp["rent_psf_num"] = pd.to_numeric(
        temp["rent_psf"]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("$", "", regex=False)
        .str.strip(),
        errors="coerce",
    )

    all_units.append(temp[["region", "size_num", "rent_psf_num"]])
    
if all_units:
    all_units_df = pd.concat(all_units, ignore_index=True).dropna(subset=["size_num", "rent_psf_num"])

    pearson_corr = all_units_df["size_num"].corr(all_units_df["rent_psf_num"], method="pearson")
    print(pearson_corr)
else:
    print("No usable 'size' and 'rent_psf' columns found in region_data.")

There is a weak negative linear relationship between unit size and rent_psf: In general, larger units tend to have slightly lower rent per square foot, but with a lot of variability.


Footfall score:\
Scale 0-100, randomized in python.\
Tier 1 – CBD : footfall_score ~ Normal(mean=80, std=10)\
Tier 2 – heartland : footfall_score ~ Normal(mean=60, std=10)

OR make use of tier, mrt_distance, size, rent_psf variables

potential = a0 + a1 * footfall_score + a2 * floor_score + e.\
revenue_psf = b0 + b1 * potential + n.\

Revenue of each unit = size * revenue_psf\
Rent of each unit = size * rent_psf\
Margin of each unit = revenue – rent\
revenue-to-rent ratio  of each unit = revenue / rent


beta_rent: how sensitive the score is to changes in rent_psf(After taking logs and standardisation).   
higher beta_rent -> differences in rent_psf matter more for score\
if beta_rent is 0, rent_psf has not impact

beta_size: how sensitive score is to unit size(in standardised form)\
higher beta_size -> larger units get biggest boost/penalty if negative\
if beta_size is 0, size does not affect the score

## generate footfall_score based on tier, mrt_distance, rent_psf, size with noise


In [ ]:
# generate footfall_score based on tier, mrt_distance, rent_psf, size with noise
def add_footfall_score(df, region_name="", beta_rent_psf=8.0, beta_size=5.0, noise_std=10.0):
    df = df.copy()

    # Ensure expected columns are numeric/string-cleaned
    tier = df["tier"].astype(str).str.strip()
    mrt_distance = pd.to_numeric(df["mrt_distance"], errors="coerce")
    rent_psf = pd.to_numeric(df["rent_psf"], errors="coerce")
    size = pd.to_numeric(df["size"], errors="coerce")

    # Fill missing numeric values so every row can get a score
    mrt_distance = mrt_distance.fillna(mrt_distance.median())
    rent_psf = rent_psf.fillna(rent_psf.median())
    size = size.fillna(size.median())

    # Region-specific tuning for South/Central: close to 100 but less hard-clipping at 100
    region_key = str(region_name).strip().lower()
    is_south_central = ("south" in region_key) or ("central" in region_key)

    # Tier base: CBD vs heartland/others (higher baseline for South/Central)
    if is_south_central:
        base_tier = np.where(tier == "CBD", 92, 82)
        local_noise_std = noise_std + 6.0  # add extra spread for these regions
        max_cap = 98.5                    # reduce pile-up at exact 100
    else:
        base_tier = np.where(tier == "CBD", 80, 50)
        local_noise_std = noise_std
        max_cap = 100

    # MRT distance band effect
    base_mrt = np.select(
        [mrt_distance <= 200, mrt_distance <= 500, mrt_distance <= 1000, mrt_distance > 1000],
        [30, 20, -10, -5],
        default=0,
    )

    # Standardize numeric features (safe if std is 0)
    log_rent = np.log(rent_psf + 1)
    log_rent_std = (log_rent - log_rent.mean()) / (log_rent.std() if log_rent.std() not in [0, np.nan] else 1)
    size_std = (size - size.mean()) / (size.std() if size.std() not in [0, np.nan] else 1)
    log_rent_std = np.nan_to_num(log_rent_std, nan=0.0)
    size_std = np.nan_to_num(size_std, nan=0.0)

    # Linear scoring + noise
    noise = np.random.normal(0, local_noise_std, size=len(df))
    score = base_tier + base_mrt + beta_rent_psf * log_rent_std + beta_size * size_std + noise

    # Keep range between 0 and region-specific cap
    df["footfall_score"] = np.clip(score, 0, max_cap)
    return df


# Add footfall_score to every row in every sheet loaded from the Excel (cell 3)
required_cols = {"tier", "mrt_distance", "rent_psf", "size"}
updated_region_data = {}

for region, sheet_df in region_data.items():
    temp_df = sheet_df.copy()

    # Ensure the output column exists for every sheet
    if required_cols.issubset(temp_df.columns):
        temp_df = add_footfall_score(temp_df, region_name=region)
    else:
        temp_df["footfall_score"] = np.nan

    updated_region_data[region] = temp_df

# overwrite the original dictionary with updated sheets
region_data = updated_region_data

# Save all updated sheets to a new Excel file
output_file = "guru1.xlsx"
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    for region, df in region_data.items():
        df.to_excel(writer, sheet_name=str(region)[:31], index=False)

print(f"Saved updated dataset with footfall_score to {output_file}")

# quick preview
for region, df in region_data.items():
    print(f"{region}: footfall_score added ({df['footfall_score'].notna().sum()}/{len(df)} rows non-null)")
display(region_data[next(iter(region_data))].head())

Floor score:\
Level 1: floor_score = 1\
Level 2: floor_score = 0.7\
level 3 and above: floor_score = 0.5\
level -1 and below = 0.6

## generate floor_score for each unit


In [ ]:
# generate floor_score using guru1.xlsx
input_file = "guru1.xlsx"
sheets = pd.read_excel(input_file, sheet_name=None)

def floor_to_score(floor_value):
    floor_num = pd.to_numeric(floor_value, errors="coerce")
    if pd.isna(floor_num):
        return np.nan
    floor_num = int(floor_num)
    if floor_num == 1:
        return 1.0
    if floor_num == 2:
        return 0.7
    if floor_num >= 3:
        return 0.5
    if floor_num <= -1:
        return 0.6
    return np.nan

for sheet_name, df in sheets.items():
    out = df.copy()
    out["floor_score"] = out["floor"].apply(floor_to_score) if "floor" in out.columns else np.nan
    sheets[sheet_name] = out

with pd.ExcelWriter(input_file, engine="openpyxl") as writer:
    for sheet_name, df in sheets.items():
        df.to_excel(writer, sheet_name=str(sheet_name)[:31], index=False)

print(f"Updated {input_file} with floor_score.")

## generate potential variable for each unit


In [ ]:
# Create potential variable for each unit in guru1.xlsx
input_file = "guru1.xlsx"
sheets = pd.read_excel(input_file, sheet_name=None)

def add_potential(df, a0=10, a1=0.5, a2=5, a3=-0.01, noise_std=5):
    out = df.copy()
    required = {"footfall_score", "floor_score", "mrt_distance"}
    if not required.issubset(out.columns):
        out["potential"] = np.nan
        return out

    footfall = pd.to_numeric(out["footfall_score"], errors="coerce")
    floor_score = pd.to_numeric(out["floor_score"], errors="coerce")
    mrt_distance = pd.to_numeric(out["mrt_distance"], errors="coerce")
    e = np.random.normal(0, noise_std, size=len(out))

    out["potential"] = a0 + a1 * footfall + a2 * floor_score + a3 * mrt_distance + e
    return out

for sheet_name, df in sheets.items():
    sheets[sheet_name] = add_potential(df)

with pd.ExcelWriter(input_file, engine="openpyxl") as writer:
    for sheet_name, df in sheets.items():
        df.to_excel(writer, sheet_name=str(sheet_name)[:31], index=False)

print(f"Updated {input_file} with potential column.")

## generate revenue_psf variable for each unit


In [ ]:
# generate revenue_psf variable for each unit in guru1.xlsx
input_file = "guru1.xlsx"
sheets = pd.read_excel(input_file, sheet_name=None)

def add_revenue_psf(df, b0=5, b1=0.5, noise_std=5):
    out = df.copy()
    if "potential" not in out.columns:
        out["revenue_psf"] = np.nan
        return out

    potential = pd.to_numeric(out["potential"], errors="coerce")
    if potential.isna().all():
        potential = pd.Series(0.0, index=out.index)
    else:
        potential = potential.fillna(potential.median())

    n = np.random.normal(0, noise_std, size=len(out))
    revenue_psf = b0 + b1 * potential + n
    out["revenue_psf"] = np.where(revenue_psf < 0, 5, revenue_psf)
    return out

for sheet_name, df in sheets.items():
    sheets[sheet_name] = add_revenue_psf(df)

with pd.ExcelWriter(input_file, engine="openpyxl") as writer:
    for sheet_name, df in sheets.items():
        df.to_excel(writer, sheet_name=str(sheet_name)[:31], index=False)

print(f"Updated {input_file} with revenue_psf column.")

## calculate occupancy cost ratio = rent_month / revenue_month * 100 for each unit


In [ ]:
# calculate occupancy cost ratio = rent_month / revenue_month * 100 for each unit in guru1.xlsx
import pandas as pd
import numpy as np

input_file = "guru1.xlsx"
sheets = pd.read_excel(input_file, sheet_name=None)

threshold = 25.0

updated_sheets = {}
all_rows = []

for region_name, df in sheets.items():
    out = df.copy()
    out["region"] = region_name

    rent_month = pd.to_numeric(out["rent_month"], errors="coerce") if "rent_month" in out.columns else pd.Series(np.nan, index=out.index)
    revenue_month = pd.to_numeric(out["revenue_month"], errors="coerce") if "revenue_month" in out.columns else pd.Series(np.nan, index=out.index)

    out["occ_ratio_pct"] = np.where(revenue_month > 0, 100 * rent_month / revenue_month, np.nan)
    out["occ_label"] = np.where(
        out["occ_ratio_pct"] <= threshold,
        "good_occ_cost",
        np.where(out["occ_ratio_pct"] >= threshold, "poor_occ_cost", "mid"),
    )

    updated_sheets[region_name] = out
    all_rows.append(out[["region", "occ_ratio_pct", "occ_label", "revenue_month"]])

df_valid = pd.concat(all_rows, ignore_index=True)
df_valid = df_valid[pd.to_numeric(df_valid["revenue_month"], errors="coerce") > 0].copy()

#which region has lowest average, median occupancy cost ratio
region_summary = (
    df_valid.groupby("region")
    .agg(
        mean_occ_ratio_pct=("occ_ratio_pct", "mean"),
        median_occ_ratio_pct=("occ_ratio_pct", "median"),
        units=("occ_ratio_pct", "size"),
    )
    .reset_index()
    .sort_values("mean_occ_ratio_pct")
)

#for each region, how many units are good vs poor occupancy cost
region_quality = (
    df_valid.groupby(["region", "occ_label"])["occ_ratio_pct"]
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

with pd.ExcelWriter(input_file, engine="openpyxl") as writer:
    for sheet_name, out in updated_sheets.items():
        out.to_excel(writer, sheet_name=str(sheet_name)[:31], index=False)

print(f"Updated {input_file} with occ_ratio_pct and occ_label columns.")
display(region_summary)
display(region_quality) #Identify units at risk of unsustainable rent for targeted support

Updated guru1.xlsx with occ_ratio_pct and occ_label columns.


,region,mean_occ_ratio_pct,median_occ_ratio_pct,units
3,south,41.379629,32.059493,30
4,west,63.644942,56.104980,30
0,central,65.798091,53.912293,30
1,east,78.760987,62.639425,30
2,north,81.409667,67.490198,30


occ_label,region,good_occ_cost,poor_occ_cost
0,central,6,24
1,east,2,28
2,north,2,28
3,south,7,23
4,west,6,24


In [ ]:
#Machine Learning use cases